# Домашнее задание: Retrieval-Augmented Generation (RAG)

**Курс:** NLP-2  
**Тема:** Построение, оптимизация и комплексная оценка RAG-систем.

## Введение
В рамках данного задания вам предстоит пройти полный путь ML-инженера при работе с RAG: от сборки базового пайплайна до тонкой настройки ретривера и генератора. Мы будем работать с датасетом **SciFact** (проверка научных фактов).

### Методология Эксперимента
Важно соблюдать гигиену ML-экспериментов. Мы разделили данные на два сета:
1.  **Main Split (200 запросов):** Ваша "Dev" выборка. Все промежуточные прогоны, подбор гиперпараметров (chunk size, top_k) и отладку вы делаете **только** на ней. Мы хотим избежать переобучения под тестовые данные.
2.  **Challenge Split (50 запросов):** Ваша "Test" выборка. Вы используете её **ровно один раз** в самом конце ноутбука для финальной валидации лучшей конфигурации.

---

## 1. Подготовка окружения
Установка зависимостей. Мы используем `LlamaIndex` как оркестратор, `Qdrant` как векторную БД и `HuggingFace` для моделей.

In [ ]:
%%capture
!pip install llama-index-core llama-index-llms-huggingface llama-index-embeddings-huggingface llama-index-vector-stores-qdrant llama-index-retrievers-bm25
!pip install qdrant-client ir_datasets ir_measures bitsandbytes accelerate transformers
!pip install pandas matplotlib seaborn tqdm

In [ ]:
# JAX Memory Configuration (CRITICAL: must run before imports)
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
print("JAX configured")

In [ ]:
import os
import time
import random
import gc
from dataclasses import dataclass, field
from typing import List, Optional, Dict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Optimizations for T4 GPU in Colab
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ARTIFACTS_DIR = "/content/artifacts"
QDRANT_PATH = f"{ARTIFACTS_DIR}/qdrant_local"
LOGS_DIR = f"{ARTIFACTS_DIR}/logs"

os.makedirs(QDRANT_PATH, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

## 2. Загрузка данных (SciFact)
Загружаем корпус и запросы. SciFact — это сложный датасет, где запрос часто требует понимания причинно-следственных связей в научных текстах.

In [ ]:
import ir_datasets

ds_corpus = ir_datasets.load("beir/scifact")
ds_test = ir_datasets.load("beir/scifact/test")

print("Loading corpus...")
corpus_rows = [{"doc_id": str(d.doc_id), "title": d.title or "", "text": d.text or ""} for d in ds_corpus.docs_iter()]
df_corpus = pd.DataFrame(corpus_rows)

print("Loading queries...")
query_rows = [{"query_id": str(q.query_id), "text": q.text} for q in ds_test.queries_iter()]
df_queries = pd.DataFrame(query_rows)

print("Loading qrels...")
qrel_rows = [{"query_id": str(q.query_id), "corpus_id": str(q.doc_id), "score": int(q.relevance)} for q in ds_test.qrels_iter()]
df_qrels = pd.DataFrame(qrel_rows)

# Map for metrics calculation
qrels_map = df_qrels.groupby("query_id").apply(lambda x: dict(zip(x["corpus_id"], x["score"]))).to_dict()

# Fixed Splits
all_qids = df_queries["query_id"].unique()
rng = np.random.default_rng(42)
rng.shuffle(all_qids)
split_main = all_qids[:200]
split_challenge = all_qids[200:250]

print(f"Corpus size: {len(df_corpus)}")
print(f"Main split: {len(split_main)} queries, Challenge split: {len(split_challenge)} queries")

## 3. Конфигурация и Модели
Мы используем `dataclasses` для строгой типизации конфигов экспериментов.

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from qdrant_client import QdrantClient
from transformers import BitsAndBytesConfig

# --- CONFIG CLASSES ---
@dataclass
class ChunkingConfig:
    chunk_size: int = 512
    chunk_overlap: int = 100
    use_hierarchical: bool = False
    child_chunk_size: int = 128

@dataclass
class RetrievalConfig:
    top_k: int = 5          # Final k documents for LLM
    overfetch_k: int = 15   # Candidates fetched from DB (The "Funnel" top)
    mode: str = "dense"     # "dense" | "hybrid"
    use_reranker: bool = False
    rerank_top_n: int = 5   # Documents to keep after reranking

@dataclass
class LLMConfig:
    model_name: str = "Qwen/Qwen3-4B-Instruct-2507"
    max_new_tokens: int = 256
    context_window: int = 2048
    load_in_4bit: bool = True
    # Subsample for E2E evaluation speed
    e2e_eval_n: int = 10
    prompt_template: str = "Context:\n{context_str}\n\nQuery: {query_str}\nAnswer:"

@dataclass
class EmbeddingConfig:
    model_name: str = "Qwen/Qwen3-Embedding-0.6B"
    truncate_dim: Optional[int] = None # For Matryoshka learning

@dataclass
class RAGConfig:
    name: str = "baseline"
    chunking: ChunkingConfig = field(default_factory=ChunkingConfig)
    retrieval: RetrievalConfig = field(default_factory=RetrievalConfig)
    llm: LLMConfig = field(default_factory=LLMConfig)
    embedding: EmbeddingConfig = field(default_factory=EmbeddingConfig)
    qdrant_collection: str = "scifact_dense_base"
    recreate_collection: bool = False
    rerank_model: str = "Qwen/Qwen3-Reranker-0.6B"

# --- MODEL HELPERS ---
_CACHED_LLM = None
_CACHED_EMBED = None

def unload_embedder():
    global _CACHED_EMBED
    if _CACHED_EMBED:
        del _CACHED_EMBED
        _CACHED_EMBED = None
        Settings.embed_model = None
        gc.collect(); torch.cuda.empty_cache()

def get_embedder(cfg: EmbeddingConfig):
    global _CACHED_EMBED
    if _CACHED_EMBED: return _CACHED_EMBED
    _CACHED_EMBED = HuggingFaceEmbedding(
        model_name=cfg.model_name,
        device="cuda",
        normalize=True,
        truncate_dim=cfg.truncate_dim
    )
    return _CACHED_EMBED

def get_llm(cfg: LLMConfig):
    global _CACHED_LLM
    if _CACHED_LLM: return _CACHED_LLM
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
    _CACHED_LLM = HuggingFaceLLM(
        model_name=cfg.model_name,
        context_window=cfg.context_window,
        max_new_tokens=cfg.max_new_tokens,
        model_kwargs={"quantization_config": bnb},
        generate_kwargs={"do_sample": False},
        device_map="auto"
    )
    return _CACHED_LLM

## 4. Система Метрик (Metrics System)

Для комплексной оценки качества мы используем двухуровневую систему метрик.

### 1. Retrieval Phase (Поиск)
Оценивает, насколько релевантные куски текста (chunks) мы нашли в базе.

**A. С учителем (Supervised / Reference-based):**
Требуют наличия "правильных ответов" (qrels).
*   **`nDCG@k`**: Основная метрика ранжирования. Учитывает не только факт нахождения правильного документа, но и его позицию (чем выше, тем лучше).
*   **`Recall@k`**: Полнота. Какую долю всех релевантных документов мы нашли в топ-k.

**B. Без учителя (Unsupervised / Reference-free):**
Полезны для мониторинга в продакшене, где нет разметки.
*   **`Mean Relevance Score`**: Среднее значение скора (similarity), который выдает ретривер. Показывает "уверенность" модели в найденном.
*   **`Redundancy`**: Избыточность. Показывает, насколько найденные чанки похожи друг на друга. Высокая избыточность — плохо (мы хотим разнообразный контекст).

### 2. Generation Phase (Генерация)
Оценивает качество финального ответа с использованием **LLM-as-a-Judge** (одна LLM оценивает другую).

*   **`Faithfulness` (Answer vs Context):** Верность контексту. Проверяет, что ответ сгенерирован **исключительно** на основе найденных документов, без галлюцинаций.
*   **`Answer Relevancy` (Answer vs Query):** Релевантность вопроса. Проверяет, что сгенерированный текст действительно отвечает на поставленный вопрос пользователя.

In [ ]:
import ir_measures
from ir_measures import nDCG, Recall, RR

RUNS_CSV_PATH = os.path.join(LOGS_DIR, "runs.csv")

def compute_redundancy(contexts: List[str]) -> float:
    """Calculates semantic redundancy (higher = more duplicate info)."""
    if not contexts: return 0.0
    unique = len(set(contexts))
    return 1.0 - (unique / len(contexts))

def compute_retrieval_metrics(qrels_map, run_doc_ids, run_contexts, run_scores, k=5):
    # 1. Reference-based Metrics (Ground Truth needed)
    run_ir = {}
    for qid, doc_ids in run_doc_ids.items():
        run_ir[qid] = {doc_id: float(-(rank + 1)) for rank, doc_id in enumerate(doc_ids[:k])}

    measures = [nDCG@k, Recall@k]
    agg = ir_measures.calc_aggregate(measures, qrels_map, run_ir)

    # 2. Reference-free Metrics (No Ground Truth)
    red_vals = [compute_redundancy(ctxs[:k]) for ctxs in run_contexts.values()]
    # Mean Score: How confident is the retriever?
    score_vals = [np.mean(scores[:k]) for scores in run_scores.values() if scores]

    return {
        "ndcg@k": float(agg[nDCG@k]),
        "recall@k": float(agg[Recall@k]),
        "ref_free_mean_score": float(np.mean(score_vals)) if score_vals else 0.0,
        "ref_free_redundancy": float(np.mean(red_vals)) if red_vals else 0.0
    }

def log_run(cfg, split, stage, run_name, metrics):
    row = {
        "ts": time.strftime("%H:%M:%S", time.gmtime()),
        "run_name": run_name,
        "split": split,
        "stage": stage,
        **metrics
    }
    df_old = pd.read_csv(RUNS_CSV_PATH) if os.path.exists(RUNS_CSV_PATH) else pd.DataFrame()
    pd.concat([df_old, pd.DataFrame([row])], ignore_index=True).to_csv(RUNS_CSV_PATH, index=False)

def show_leaderboard(stage="retrieval", split="main"):
    if not os.path.exists(RUNS_CSV_PATH): return
    df = pd.read_csv(RUNS_CSV_PATH)
    df_sub = df[(df["stage"]==stage) & (df["split"]==split)].copy()
    if df_sub.empty: return

    # Sort by key metric
    sort_col = "ndcg@k" if stage == "retrieval" else "faithfulness"
    if sort_col in df_sub.columns:
        df_sub = df_sub.sort_values(sort_col, ascending=False)

    # Select columns to display
    cols_ret = ["run_name", "ndcg@k", "recall@k", "ref_free_mean_score", "ref_free_redundancy", "latency_s"]
    cols_gen = ["run_name", "faithfulness", "answer_relevancy", "latency_s"]

    cols = cols_ret if stage == "retrieval" else cols_gen
    print(f"\n🏆 LEADERBOARD [{stage.upper()} | {split.upper()}] 🏆")
    display(df_sub[[c for c in cols if c in df_sub.columns]])

def _get_df(split): return df_queries[df_queries["query_id"].isin(split_main if split=="main" else split_challenge)]

def run_retrieval(cfg: RAGConfig, split: str, run_name: str):
    unload_embedder()
    index, retriever, _ = build_rag_pipeline(cfg, df_corpus, load_llm=False)

    df_q = _get_df(split)
    results_dids, results_txts, results_scores = {}, {}, {}

    t0 = time.time()
    for row in tqdm(df_q.itertuples(index=False), total=len(df_q), desc=f"Retr {run_name}"):
        nodes = retriever.retrieve(row.text)
        # De-duplicate by doc_id to avoid metrics skew
        seen, dids, txts, scores = set(), [], [], []
        for n in nodes:
            if n.metadata["doc_id"] not in seen:
                seen.add(n.metadata["doc_id"])
                dids.append(n.metadata["doc_id"])
                txts.append(n.get_text())
                scores.append(n.score if n.score else 0.0)
        results_dids[str(row.query_id)] = dids
        results_txts[str(row.query_id)] = txts
        results_scores[str(row.query_id)] = scores

    # FIX: filter qrels_map based on current split
    split_qids = set(df_q["query_id"].astype(str))
    qrels_map_filtered = {qid: docs for qid, docs in qrels_map.items() if qid in split_qids}
    
    latency = time.time() - t0

    metrics = compute_retrieval_metrics(qrels_map_filtered, results_dids, results_txts, results_scores, k=cfg.retrieval.top_k)
    metrics["latency_s"] = latency
    log_run(cfg, split, "retrieval", run_name, metrics)
    show_leaderboard("retrieval", split)
    unload_embedder()

def run_generation(cfg: RAGConfig, split: str, run_name: str):
    """Generates answers and saves them to JSON for subsequent evaluation."""
    # 1. Retrieval Phase
    unload_embedder()
    index, retriever, _ = build_rag_pipeline(cfg, df_corpus, load_llm=False)

    df_q = _get_df(split).head(cfg.llm.e2e_eval_n)
    records = []
    for row in tqdm(df_q.itertuples(index=False), total=len(df_q), desc="Retrieving Contexts"):
        nodes = retriever.retrieve(row.text)
        ctx_list = [n.get_text() for n in nodes[:cfg.retrieval.top_k]]
        records.append({"query_id": str(row.query_id), "q": row.text, "ctx": "\n".join(ctx_list)})

    unload_embedder()

    # 2. Generation Phase
    Settings.embed_model = None
    llm = get_llm(cfg.llm)

    t0 = time.time()
    for item in tqdm(records, desc="Generating"):
        prompt = cfg.llm.prompt_template.format(context_str=item["ctx"], query_str=item["q"])
        try: 
            item["a"] = llm.complete(prompt).text
        except: 
            item["a"] = ""
    latency = time.time() - t0

    # 3. Save results
    output_path = os.path.join(LOGS_DIR, f"generations_{run_name}_{split}.json")
    import json
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump({
            "records": records, 
            "latency_s": latency, 
            "cfg_name": cfg.name,
            "run_name": run_name,
            "split": split
        }, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Generations saved to {output_path}")
    return output_path


def run_evaluation(generation_file: str, eval_name: Optional[str] = None):
    """Evaluates already generated answers."""
    import json
    
    # Load generated data
    with open(generation_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    records = data["records"]
    latency = data["latency_s"]
    split = data.get("split", "main")
    run_name = eval_name if eval_name else data.get("run_name", "eval")
    
    # Load LLM for Judge
    Settings.embed_model = None
    llm = get_llm(LLMConfig())
    
    # Judge Phase
    faith_scores = []
    rel_scores = []

    for item in tqdm(records, desc=f"Judging {run_name}"):
        if not item.get("a", ""):
            faith_scores.append(0.0); rel_scores.append(0.0)
            continue

        # Metric 1: Faithfulness (Context vs Answer)
        p_faith = f"Context: {item['ctx'][:1000]}\nAnswer: {item['a']}\nDoes the Answer use the Context? YES/NO."
        res_f = llm.complete(p_faith, max_new_tokens=5).text.upper()
        faith_scores.append(1.0 if "YES" in res_f else 0.0)

        # Metric 2: Answer Relevancy (Question vs Answer)
        p_rel = f"Question: {item['q']}\nAnswer: {item['a']}\nDoes the Answer address the Question? YES/NO."
        res_r = llm.complete(p_rel, max_new_tokens=5).text.upper()
        rel_scores.append(1.0 if "YES" in res_r else 0.0)

    metrics = {
        "faithfulness": np.mean(faith_scores),
        "answer_relevancy": np.mean(rel_scores),
        "latency_s": latency
    }
    
    log_run(RAGConfig(name=run_name), split, "e2e", run_name, metrics)
    show_leaderboard("e2e", split)
    return metrics


def run_e2e(cfg: RAGConfig, split: str, run_name: str, generation_file: Optional[str] = None):
    """
    End-to-end evaluation with option to reuse generations.
    
    Args:
        cfg: RAG configuration
        split: "main" or "challenge"
        run_name: Experiment name
        generation_file: Path to previously generated answers file (optional)
    """
    if generation_file and os.path.exists(generation_file):
        # Reuse existing generation
        print(f"📂 Loading generations from {generation_file}")
        return run_evaluation(generation_file, run_name)
    else:
        # Full cycle: generation + evaluation
        gen_file = run_generation(cfg, split, run_name)
        return run_evaluation(gen_file, run_name)


## Task 0: Реализация Baseline Pipeline (4 балла)

Ваша первая задача — реализовать функцию `build_rag_pipeline`. На данном этапе требуется собрать простую архитектуру Dense Retrieval.

**Требования к реализации:**
1.  Инициализация моделей через `Settings` (используйте `get_embedder` и `get_llm`).
2.  Подключение к `QdrantVectorStore`.
3.  Логика индексации: Если коллекция не существует (или `recreate_collection=True`), создать её из `df_corpus`, используя `SentenceSplitter`. Если существует — загрузить существующий индекс.
4.  Возврат объектов `index`, `retriever` и `query_engine`.

Обратите внимание на параметр `overfetch_k` в конфиге. Ретривер должен извлекать именно это количество кандидатов.

In [ ]:
def build_rag_pipeline(cfg: RAGConfig, df_corpus: pd.DataFrame, load_llm: bool = True):
    # 1. Setup Models
    Settings.embed_model = get_embedder(cfg.embedding)
    if load_llm: Settings.llm = get_llm(cfg.llm)

    # 2. Vector Store Setup (TODO)
    # client = ...
    # vector_store = ...
    # <YOUR CODE HERE>

    # 3. Indexing Logic (TODO)
    # Check if collection exists.
    # If yes & not recreate -> load index.
    # Else -> create documents, setup splitter, build index.
    # <YOUR CODE HERE>

    # 4. Retrieval Construction (TODO)
    # retriever = ... (use cfg.retrieval.overfetch_k)
    # query_engine = ...
    # <YOUR CODE HERE>

    # return index, retriever, query_engine
    pass

In [ ]:
# TEST YOUR BASELINE
baseline_cfg = RAGConfig(
    name="baseline",
    chunking=ChunkingConfig(chunk_size=512, chunk_overlap=50),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    qdrant_collection="scifact_base_512"
)

print("Running Baseline...")
# 1. Run Retrieval
# TODO: Uncomment after implementing pipeline
# run_retrieval(baseline_cfg, "main", "baseline")

# 2. Check Results
# show_leaderboard("retrieval", "main")

## Task 1: Chunking Strategies (2 балла)

Размер контекста критически влияет на качество поиска.

**Задание:**
1.  **Small vs Large:** Создайте конфигурации для `chunk_size=256` и `chunk_size=1024`. Запустите эксперименты. *Напоминание: при изменении чанкинга необходимо менять имя коллекции или ставить `recreate_collection=True`.*
2.  **Hierarchical Chunking:** Реализуйте иерархический чанкинг (`use_hierarchical=True`). Для этого в `build_rag_pipeline` необходимо добавить проверку конфига и использование `HierarchicalNodeParser`. Этот метод индексирует мелкие чанки, но возвращает контекст родительских (крупных) блоков.

In [ ]:
# TODO: Update build_rag_pipeline to support HierarchicalNodeParser

# TODO: Define Configs
# cfg_256 = ...
# cfg_1024 = ...
# cfg_hier = ...

# TODO: Run Experiments
# run_retrieval(cfg_256, "main", "chunk_256")
# show_leaderboard("retrieval", "main")

## Task 2: Optimization & Advanced Retrieval (3 балла)

В этом блоке мы реализуем продвинутые техники для улучшения качества и эффективности.

### 2.1 Matryoshka Embeddings (Optimization)
Модель `Qwen/Qwen3-Embedding` поддерживает **Matryoshka Representation Learning (MRL)**. Это позволяет усекать размерность векторов (например, с 1024 до 512) с минимальной потерей качества, что экономит память и ускоряет поиск.
**Задание:** Измените `truncate_dim` в `EmbeddingConfig` (например, на 512) и проверьте влияние на метрики.

### 2.2 Advanced Retrieval (Hybrid + Rerank)
Вам необходимо модифицировать `build_rag_pipeline` (или создать `build_advanced_pipeline`), добавив следующую логику:
1.  **Hybrid Search:** Если `cfg.retrieval.mode == 'hybrid'`, создайте `BM25Retriever` и объедините его с векторным через `QueryFusionRetriever` (алгоритм Reciprocal Rank Fusion).
2.  **Reranking:** Если `cfg.retrieval.use_reranker == True`, добавьте `SentenceTransformerRerank` в список `node_postprocessors` для `QueryEngine`. Реренкер должен принимать на вход топ-K кандидатов из этапа 1 (`overfetch_k`) и оставлять лучшие N (`rerank_top_n`).

**Concept:** Funnel Architecture (Broad Search -> Narrow Filtering).

In [ ]:
# TODO: Upgrade Pipeline Logic (Hybrid, Reranker)
# def build_rag_pipeline(...):
#     ...

# TODO: Run Experiments
# cfg_matryoshka = ...
# cfg_hybrid_rerank = ...
# run_retrieval(cfg_hybrid_rerank, "main", "advanced_full")
# show_leaderboard("retrieval", "main")

## Task 3: Generator Tuning (Prompt Engineering) (1 балл)

Качество ретривала (nDCG) мы оптимизировали. Теперь фокус на Генераторе.

**Задание:**
Модифицируйте `prompt_template` в `LLMConfig`. Цель — максимизировать метрики **Faithfulness** и **Answer Relevancy**.
Рекомендуемые техники:
*   **Role Prompting:** ("You are an expert scientist...")
*   **Constraint Enforcement:** ("Answer ONLY based on the context. If unsure, say 'I don't know'.")
*   **Chain-of-Thought:** ("Let's analyze the context step by step...")

In [ ]:
# TODO: Create Prompt Config
# cfg_prompt = ...
# run_e2e(cfg_prompt, "main", "prompt_tuned")
# show_leaderboard("e2e", "main")

## 4. Final Challenge

Выберите одну лучшую конфигурацию по совокупности метрик. Запустите её на отложенной выборке **Challenge Split**.
Сравните результаты с Baseline на этом же сплите.

In [ ]:
# TODO: Final Evaluation
# run_retrieval(best_cfg, "challenge", "final_submission")
# run_e2e(best_cfg, "challenge", "final_submission")

print("FINAL RESULTS:")
show_leaderboard("retrieval", "challenge")
show_leaderboard("e2e", "challenge")